# Anonymize Previous Files

Take old research files and create subject level data.

## Libraries

In [121]:
from pathlib import Path
import pandas as pd
import hashlib
import re

def anonymize(worker_id):
    return hashlib.sha256(str(worker_id).encode()).hexdigest()[:10]

def extract_index(col_name):
    match = re.search(r'(\d+)$', col_name)
    return int(match.group(1)) if match else None

# -------------------
# core function
# -------------------

def process_mturk_file(filepath):
    df = pd.read_csv(filepath)
    
    # detect columns
    input_cols = [c for c in df.columns if c.startswith("Input.word")]
    answer_cols = [c for c in df.columns if c.startswith("Answer.word")]
    
    # sort them properly (word1, word2, ...)
    input_cols = sorted(input_cols, key=extract_index)
    answer_cols = sorted(answer_cols, key=extract_index)
    
    rows = []
    
    for _, row in df.iterrows():
        worker_id = anonymize(row["WorkerId"])
        
        for input_col, answer_col in zip(input_cols, answer_cols):
            word = row.get(input_col)
            answer = row.get(answer_col)
            
            if pd.isna(word) or pd.isna(answer):
                continue
            
            rows.append({
                "subject_id": worker_id,
                "word": str(word).strip(),
                "answer": str(answer).strip()
            })
    
    return pd.DataFrame(rows)

def find_index(lines, keyword):
    for i, l in enumerate(lines):
        if keyword in l.lower():
            return i
    return None

def get_next_valid_lines(lines, start_idx, n=2):
    collected = []
    i = start_idx + 1
    
    while i < len(lines) and len(collected) < n:
        if lines[i].strip():
            collected.append(lines[i])
        i += 1
    
    return collected

def process_text_file(filepath, subject_id):
    rows = []
    
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        lines = [l.strip() for l in f.readlines()]
    
    # --- find sections ---
    practice_idx = find_index(lines, "practice")
    experimental_idx = find_index(lines, "experimental")
    
    # --- get practice lines (2 only) ---
    if practice_idx is not None:
        practice_lines = get_next_valid_lines(lines, practice_idx, 2)
    else:
        practice_lines = []
        print(f"⚠️ No Practice section: {filepath}")
    
    # --- get experimental lines ---
    if experimental_idx is not None:
        experimental_lines = [
            l for l in lines[experimental_idx + 1:] if l.strip()
        ]
    else:
        experimental_lines = []
        print(f"⚠️ No Experimental section: {filepath}")
    
    all_lines = practice_lines + experimental_lines
    
    # --- parse lines ---
    for line in all_lines:
        parts = line.split(" ", 1)
        
        if len(parts) < 2:
            continue
        
        word, answer = parts
        
        rows.append({
            "subject_id": subject_id,
            "word": word.strip(),
            "answer": answer.strip()
        })
    
    return rows

def process_all_subjects(root_dir, output_file):
    all_rows = []
    subject_counter = 1
    
    root = Path(root_dir)
    
    # handle both: parent dir OR single indv_subs folder
    if root.name.startswith("indv_subs_"):
        folders = [root]
    else:
        folders = sorted(root.glob("indv_subs_*"))
    
    for folder in folders:
        print(f"Processing folder: {folder}")
        
        for file in folder.glob("*.txt"):
            # print(f"  → {file.name}")
            
            rows = process_text_file(file, subject_counter)
            all_rows.extend(rows)
            
            subject_counter += 1
    
    df = pd.DataFrame(all_rows)
    df.to_csv(output_file, index=False)
    
    print(f"\n✅ Saved {len(df)} rows from {subject_counter-1} subjects")
    return df

def process_word_list_csv_file(filepath):
    df = pd.read_csv(filepath, header=None, names=["word", "answer"])
    
    df["word"] = df["word"].astype(str).str.strip()
    df["answer"] = df["answer"].astype(str).str.strip()
    
    df = df[df["answer"] != ""]
    df = df[~df["answer"].isna()]
    
    df["subject_id"] = df.groupby("word").cumcount() + 1
    
    return df[["subject_id", "word", "answer"]]

def process_word_list_txt_file(filepath):
    rows = []
    
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        lines = [l.strip() for l in f.readlines()]
    
    for line in lines:
        if not line:
            continue
        
        parts = line.split(None, 1)  # split on first whitespace
        
        if len(parts) < 2:
            continue
        
        word, answer = parts
        
        rows.append({
            "word": word.strip(),
            "answer": answer.strip()
        })
    
    df = pd.DataFrame(rows)
    
    # assign subject IDs per word
    df["subject_id"] = df.groupby("word").cumcount() + 1
    
    return df[["subject_id", "word", "answer"]]

def process_word_list_folder(folder_path, output_file=None):
    all_rows = []
    subject_offset = 0  # ensures unique IDs across files
    
    folder = Path(folder_path)
    
    for file in sorted(folder.glob("*")):
        print(f"Processing: {file.name}")
        
        if file.suffix.lower() == ".csv":
            df = process_word_list_csv_file(file)
        
        elif file.suffix.lower() == ".txt":
            df = process_word_list_txt_file(file)
        
        else:
            print(f"  ⚠️ Skipping unsupported file: {file.name}")
            continue
        
        # shift subject IDs so they don't overlap across files
        df["subject_id"] += subject_offset
        
        subject_offset = df["subject_id"].max()
        
        all_rows.append(df)
    
    final_df = pd.concat(all_rows, ignore_index=True)
    
    if output_file:
        final_df.to_csv(output_file, index=False)
    
    print(f"\n✅ Total rows: {len(final_df)}")
    print(f"✅ Total subjects: {final_df['subject_id'].nunique()}")
    
    return final_df

def process_wide_wordlist(
    filepath,
    output_path=None,
    id_col="id",
    drop_cols=None
):
    df = pd.read_csv(filepath)
    
    # --- columns to ignore ---
    drop_cols = drop_cols or []
    
    # ensure id exists
    if id_col not in df.columns:
        raise ValueError(f"Column '{id_col}' not found")
    
    # columns that are words
    word_cols = [
        c for c in df.columns
        if c != id_col and c not in drop_cols
    ]
    
    # --- reshape ---
    long_df = df.melt(
        id_vars=[id_col],
        value_vars=word_cols,
        var_name="word",
        value_name="answer"
    )
    
    # clean
    long_df["answer"] = long_df["answer"].astype(str).str.strip()
    
    # drop empty answers
    long_df = long_df[long_df["answer"] != ""]
    long_df = long_df[~long_df["answer"].isna()]
    
    # rename id → subject_id
    long_df = long_df.rename(columns={id_col: "subject_id"})
    
    # reorder
    long_df = long_df[["subject_id", "word", "answer"]]
    
    if output_path:
        long_df.to_csv(output_path, index=False)
    
    print(f"✅ Processed {len(long_df)} rows")
    return long_df

def process_paired_wide_no_id(filepath, output_path=None, start_id=1):
    df = pd.read_csv(filepath)
    
    # assign subject IDs
    df["subject_id"] = range(start_id, start_id + len(df))
    
    # --- find word/answer columns correctly ---
    word_cols = []
    answer_cols = []
    
    for col in df.columns:
        col_lower = col.lower()
        
        if "input.word" in col_lower:
            word_cols.append(col)
        elif "answer.word" in col_lower:
            answer_cols.append(col)
    
    # sort by number
    def extract_num(col):
        m = re.search(r'(\d+)', col)
        return int(m.group(1)) if m else 0
    
    word_cols = sorted(word_cols, key=extract_num)
    answer_cols = sorted(answer_cols, key=extract_num)
    
    # --- build long format ---
    rows = []
    
    for _, row in df.iterrows():
        subject_id = row["subject_id"]
        
        for w_col, a_col in zip(word_cols, answer_cols):
            word = row[w_col]
            answer = row[a_col]
            
            if pd.isna(word) or pd.isna(answer):
                continue
            
            word = str(word).strip()
            answer = str(answer).strip()
            
            if not word or not answer:
                continue
            
            rows.append({
                "subject_id": subject_id,
                "word": word,
                "answer": answer
            })
    
    long_df = pd.DataFrame(rows)
    
    if output_path:
        long_df.to_csv(output_path, index=False)
    
    print(f"✅ Processed {len(long_df)} rows from {len(df)} subjects")
    return long_df

def merge_all_processed_data(
    input_dir="raw_data",
    output_file="processed/all_data.csv"
):
    all_dfs = []
    subject_offset = 0
    
    input_path = Path(input_dir)
    
    for file in sorted(input_path.glob("*.csv")):
        print(f"Processing: {file.name}")
        
        df = pd.read_csv(file)
        
        # --- basic validation ---
        required_cols = {"subject_id", "word", "answer"}
        if not required_cols.issubset(df.columns):
            print(f"  ⚠️ Skipping (missing required columns): {file.name}")
            continue
        
        # --- offset subject IDs ---
        # --- remap subject IDs correctly ---
        unique_subjects = df["subject_id"].astype(str).unique()

        mapping = {
            old_id: new_id
            for new_id, old_id in enumerate(
                unique_subjects,
                start=subject_offset + 1
            )
        }

        df["subject_id"] = df["subject_id"].astype(str).map(mapping)

        # ✅ FIX: ensure offset stays numeric
        subject_offset = int(df["subject_id"].max())
        
        # --- track source ---
        df["source_file"] = file.name
        
        # --- ID note ---
        df["id_note"] = "subject_ids_unique_within_file_not_global"
        df["turk_note"] = "word_list_5_long turk ids are not all individual subjects, but linking is lost to time"
        
        # update offset
        subject_offset = df["subject_id"].max()
        
        all_dfs.append(df)
    
    # --- combine ---
    final_df = pd.concat(all_dfs, ignore_index=True)
    
    # save
    Path(output_file).parent.mkdir(exist_ok=True)
    final_df.to_csv(output_file, index=False)
    
    print("\n✅ FINAL SUMMARY")
    print(f"Rows: {len(final_df)}")
    print(f"Subjects: {final_df['subject_id'].nunique()}")
    print(f"Files merged: {len(all_dfs)}")
    
    return final_df

## Data

### Mturk Files 

Raw Mturk files with ids to clean up.

In [122]:
input_file = "raw_data_nosync/mturk_raw/mturk raw 1.csv"
output_file = "raw_data/mturk_file_1.csv"
    
Path("processed").mkdir(exist_ok=True)
    
df = process_mturk_file(input_file)
df.to_csv(output_file, index=False)
    
print(f"Saved {len(df)} rows to {output_file}")

Saved 5925 rows to raw_data/mturk_file_1.csv


In [123]:
input_file = "raw_data_nosync/mturk_raw/mturk raw 2.csv"
output_file = "raw_data/mturk_file_2.csv"
    
Path("processed").mkdir(exist_ok=True)
    
df = process_mturk_file(input_file)
df.to_csv(output_file, index=False)
    
print(f"Saved {len(df)} rows to {output_file}")

Saved 1579 rows to raw_data/mturk_file_2.csv


In [124]:
input_file = "raw_data_nosync/mturk_raw/mturk raw 3.csv"
output_file = "raw_data/mturk_file_3.csv"
    
Path("processed").mkdir(exist_ok=True)
    
df = process_mturk_file(input_file)
df.to_csv(output_file, index=False)
    
print(f"Saved {len(df)} rows to {output_file}")

Saved 2599 rows to raw_data/mturk_file_3.csv


In [125]:
input_file = "raw_data_nosync/mturk_raw/mturk raw 4.csv"
output_file = "raw_data/mturk_file_4.csv"
    
Path("processed").mkdir(exist_ok=True)
    
df = process_mturk_file(input_file)
df.to_csv(output_file, index=False)
    
print(f"Saved {len(df)} rows to {output_file}")

Saved 359 rows to raw_data/mturk_file_4.csv


In [126]:
input_file = "raw_data_nosync/mturk_raw/mturk raw 5.csv"
output_file = "raw_data/mturk_file_5.csv"
    
Path("processed").mkdir(exist_ok=True)
    
df = process_mturk_file(input_file)
df.to_csv(output_file, index=False)
    
print(f"Saved {len(df)} rows to {output_file}")

Saved 300 rows to raw_data/mturk_file_5.csv


In [127]:
input_file = "raw_data_nosync/mturk_raw/mturk raw 6.csv"
output_file = "raw_data/mturk_file_6.csv"
    
Path("processed").mkdir(exist_ok=True)
    
df = process_mturk_file(input_file)
df.to_csv(output_file, index=False)
    
print(f"Saved {len(df)} rows to {output_file}")

Saved 395 rows to raw_data/mturk_file_6.csv


In [128]:
input_file = "raw_data_nosync/mturk_raw/mturk raw 7.csv"
output_file = "raw_data/mturk_file_7.csv"
    
Path("processed").mkdir(exist_ok=True)
    
df = process_mturk_file(input_file)
df.to_csv(output_file, index=False)
    
print(f"Saved {len(df)} rows to {output_file}")

Saved 2108 rows to raw_data/mturk_file_7.csv


In [129]:
input_file = "raw_data_nosync/mturk_raw/mturk raw 8.csv"
output_file = "raw_data/mturk_file_8.csv"
    
Path("processed").mkdir(exist_ok=True)
    
df = process_mturk_file(input_file)
df.to_csv(output_file, index=False)
    
print(f"Saved {len(df)} rows to {output_file}")

Saved 6974 rows to raw_data/mturk_file_8.csv


In [130]:
input_file = "raw_data_nosync/mturk_raw/mturk raw 9.csv"
output_file = "raw_data/mturk_file_9.csv"
    
Path("processed").mkdir(exist_ok=True)
    
df = process_mturk_file(input_file)
df.to_csv(output_file, index=False)
    
print(f"Saved {len(df)} rows to {output_file}")

Saved 4493 rows to raw_data/mturk_file_9.csv


### Individual Subject Files

In [131]:
process_all_subjects(
    root_dir="raw_data_nosync",
    output_file="raw_data/individual_subjects.csv"
)

Processing folder: raw_data_nosync/indv_subs_1
⚠️ No Practice section: raw_data_nosync/indv_subs_1/s43.txt
Processing folder: raw_data_nosync/indv_subs_10
Processing folder: raw_data_nosync/indv_subs_11
Processing folder: raw_data_nosync/indv_subs_2
Processing folder: raw_data_nosync/indv_subs_3
Processing folder: raw_data_nosync/indv_subs_4
Processing folder: raw_data_nosync/indv_subs_5
Processing folder: raw_data_nosync/indv_subs_6
Processing folder: raw_data_nosync/indv_subs_7
Processing folder: raw_data_nosync/indv_subs_8
Processing folder: raw_data_nosync/indv_subs_9

✅ Saved 46806 rows from 766 subjects


,subject_id,word,answer
0,1,finger,"foods, paint, hands"
1,1,careless,free flowing
2,1,casual,"laid back, calm, jeans and a tshirt"
3,1,calories,food
4,1,checking,check book
...,...,...,...
46801,766,56,"cabbage greens, bowling, water, mom, cooking, ..."
46802,766,57,"feet toes, socks, ankle, shoes, toenails, sandals"
46803,766,58,"quiet shhhh!, no noise, quiet game, silence,"
46804,766,59,"search find, look for, found, lost, wonder,"


### Process Long Files

In [132]:
df = process_word_list_folder(
    folder_path="raw_data_nosync/long_raw",
    output_file="raw_data/word_long_all.csv"
)

Processing: .DS_Store
  ⚠️ Skipping unsupported file: .DS_Store
Processing: word_list_11.csv
Processing: word_list_14.txt
Processing: word_list_15.txt
Processing: word_list_16.txt
Processing: word_list_17.txt
Processing: word_list_18.txt
Processing: word_list_19_montana.csv
Processing: word_list_1_sona.csv
Processing: word_list_20a.csv
Processing: word_list_20b.csv
Processing: word_list_21a.csv
Processing: word_list_21b.csv
Processing: word_list_22a.csv
Processing: word_list_22b.csv
Processing: word_list_23a.csv
Processing: word_list_24a.csv
Processing: word_list_24b.csv
Processing: word_list_25a.csv
Processing: word_list_25b.csv
Processing: word_list_26a.csv
Processing: word_list_2_sona.csv
Processing: word_list_3_sona.csv

✅ Total rows: 48642
✅ Total subjects: 1258


### Process Wide Files

In [133]:
df = process_wide_wordlist(
    "raw_data_nosync/wide_raw/word_list_10.csv",
    output_path="raw_data/word_list_10_long.csv",
    id_col="id_code",                # or whatever it's called
    drop_cols=["start_time","end_time","duration"]      # optional
)

✅ Processed 3534 rows


In [134]:
df = process_wide_wordlist(
    "raw_data_nosync/wide_raw/word_list_12.csv",
    output_path="raw_data/word_list_12_long.csv",
    id_col="id_code",                # or whatever it's called
    drop_cols=["start_time","end_time","duration"]      # optional
)

✅ Processed 4712 rows


In [135]:
df = process_wide_wordlist(
    "raw_data_nosync/wide_raw/word_list_13.csv",
    output_path="raw_data/word_list_13_long.csv",
    id_col="id_code",                # or whatever it's called
    drop_cols=["start_time","end_time","duration"]      # optional
)

✅ Processed 4340 rows


In [136]:
df = process_paired_wide_no_id(
    "raw_data_nosync/mturk_extra/wl5 mturk.csv",
    output_path="raw_data/word_list_5_long.csv"
)

✅ Processed 52591 rows from 10523 subjects


## Put Together Data


In [137]:
df = merge_all_processed_data(
    input_dir="raw_data",
    output_file="raw_data_all/all_data.csv"
)


Processing: individual_subjects.csv
Processing: mturk_file_1.csv
Processing: mturk_file_2.csv
Processing: mturk_file_3.csv
Processing: mturk_file_4.csv
Processing: mturk_file_5.csv
Processing: mturk_file_6.csv
Processing: mturk_file_7.csv
Processing: mturk_file_8.csv
Processing: mturk_file_9.csv
Processing: word_list_10_long.csv
Processing: word_list_12_long.csv
Processing: word_list_13_long.csv
Processing: word_list_5_long.csv
Processing: word_long_all.csv

✅ FINAL SUMMARY
Rows: 185357
Subjects: 13949
Files merged: 15
